In [2]:

import numpy as np

class MyLDA:

    def __init__(self, n_components=None):
        self.n_components = n_components

    def fit(self, X, y):

        # Number of features
        n_features = X.shape[1]

        # Find unique classes
        classes = np.unique(y)

        # Calculate overall mean
        mean_total = np.mean(X, axis=0)

        # Initialize scatter matrices
        SW = np.zeros((n_features, n_features))
        SB = np.zeros((n_features, n_features))

        # Calculate SW and SB
        for c in classes:

            # Select samples belonging to class c
            X_c = X[y == c]

            # Number of samples in this class
            n_c = X_c.shape[0]

            # Mean of this class
            mean_c = np.mean(X_c, axis=0)

            # Within-class scatter
            X_centered = X_c - mean_c
            SW += X_centered.T @ X_centered

            # Between-class scatter
            mean_diff = (mean_c - mean_total).reshape(-1, 1)

            SB += n_c * (mean_diff @ mean_diff.T)

        # Solve generalized eigenvalue problem
        eigenvalues, eigenvectors = np.linalg.eig(
            np.linalg.pinv(SW) @ SB
        )

        # Sort eigenvalues in descending order
        idx = np.argsort(eigenvalues.real)[::-1]

        eigenvalues = eigenvalues.real[idx]
        eigenvectors = eigenvectors.real[:, idx]

        # Maximum possible components = number of classes - 1
        max_components = len(classes) - 1

        if self.n_components is None:
            self.n_components = max_components

        if self.n_components > max_components:
            raise ValueError("Too many components for LDA")

        # Select top eigenvectors
        self.W = eigenvectors[:, :self.n_components]

        return self

    def transform(self, X):

        # Project original data into lower dimensions
        return X @ self.W

    def fit_transform(self, X, y):

        self.fit(X, y)

        return self.transform(X)
